In [1]:
import json
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

from datetime import datetime, timedelta
from random import randint, random

from GAMealPlanner import GAMealPlanner

from models import (
    Ingredient,
    MealPlanningEnvironment,
    NutritionalInformation,
    Pantry,
    PantryIngredient,
    UserPreferences,
)

In [2]:
preferences = UserPreferences(
    weekly_budget=50,
    calorie_target_per_day=2500,
    protein_target_per_day=80,
    is_vegetarian=False,
    is_vegan=False,
    requires_gluten_free=False,
    requires_lactose_free=False,
)

In [3]:
all_ingredients = []

with open("../recipe_extraction/supplemented_structured_ingredients.json", "r") as f:
    ingredients_data = json.load(f)

    for ingredient_data in ingredients_data:
        ingredient_nutritional_information = NutritionalInformation(**ingredient_data["nutritional_information"])

        ingredient = Ingredient(
            name=ingredient_data["name"], nutritional_information=ingredient_nutritional_information
        )
        all_ingredients.append(ingredient)

In [4]:
def get_ingredient(ingredient_name: str) -> Ingredient | None:
    return next((ingredient for ingredient in all_ingredients if ingredient.name == ingredient_name), None)

In [5]:
def get_pantry_ingredient(
    ingredient_name: str, estimated_expiration_date: datetime, estimated_financial_cost: float
) -> PantryIngredient:
    ingredient = get_ingredient(ingredient_name)

    if ingredient is None:
        raise ValueError(f"Ingredient '{ingredient_name}' not found in ingredient list")

    return PantryIngredient(
        name=ingredient.name,
        nutritional_information=ingredient.nutritional_information,
        estimated_expiration_date=estimated_expiration_date,
        estimated_financial_cost=estimated_financial_cost,
    )

In [6]:
CURRENT_DATE = datetime.now()

In [7]:
pantry_ingredient_name_by_quantity = {
    "chicken breast": 800,
    "broccoli": 1500,
    "rice": 1000,
}

In [8]:
pantry_ingredients = [
    get_pantry_ingredient(ingredient_name, CURRENT_DATE + timedelta(days=randint(1, 7)), random())
    for ingredient_name in pantry_ingredient_name_by_quantity.keys()
]

In [9]:
pantry_ingredient_by_quantity = dict(zip(pantry_ingredients, pantry_ingredient_name_by_quantity.values()))

In [10]:
pantry = Pantry()

for pantry_ingredient, quantity in pantry_ingredient_by_quantity.items():
    pantry.add(
        pantry_ingredient,
        quantity,
    )

In [11]:
pantry.print()

---
Quantity: 800
Ingredient: chicken breast
	Estimated Expiration Date: 2026-04-27
	Estimated Financial Cost per 100g: EUR 0.43
	Nutritional Information:
		Calories: 125.0 kcal
		Carbohydrates: 1.79 g
		Sugar: 0.0 g
		Protein: 16.07 g
		Fat: 5.36 g
		Saturated Fat: 1.79 g
		Fiber: 1.8 g
		Sodium: 571.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: No
		Vegan: No
---
---
Quantity: 1500
Ingredient: broccoli
	Estimated Expiration Date: 2026-04-21
	Estimated Financial Cost per 100g: EUR 0.74
	Nutritional Information:
		Calories: 31.0 kcal
		Carbohydrates: 6.27 g
		Sugar: 1.4 g
		Protein: 2.57 g
		Fat: 0.34 g
		Saturated Fat: 0.039 g
		Fiber: 2.4 g
		Sodium: 36.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: Yes
		Vegan: Yes
---
---
Quantity: 1000
Ingredient: rice
	Estimated Expiration Date: 2026-04-27
	Estimated Financial Cost per 100g: EUR 0.63
	Nutritional Information:
		Calories: 356.0 kcal
		Carbohydrates: 80.0 g
		Sugar: 0.0 g
		Protein: 6.67 g
		Fat: 0.0 g
		Satu

In [12]:
meal_planning_environment = MealPlanningEnvironment(
    recipes=[],
    pantry=pantry,
    preferences=preferences,
)

In [13]:
meal_planning_environment.load_recipes_from_json("../recipe_extraction/supplemented_structured_recipes.json")

In [14]:
planner = GAMealPlanner(meal_planning_environment)

In [15]:
NUM_GENERATIONS = 10
NUM_PARENTS_MATING = 20
POPULATION_SIZE = 100
NUM_DAYS = 7
NUM_MEALS_PER_DAY = 3
GENERATION_PRINT_INTERVAL = 10
SEED = 1

In [16]:
best_meal_plan, best_fitness_score = planner.generate_meal_plan(
    num_days=NUM_DAYS,
    meals_per_day=NUM_MEALS_PER_DAY,
    num_generations=NUM_GENERATIONS,
    num_parents_mating=NUM_PARENTS_MATING,
    population_size=POPULATION_SIZE,
    generation_print_interval=GENERATION_PRINT_INTERVAL,
    seed=SEED,
)

Generation 10, Best Fitness: -75.33


In [17]:
print(f"Best fitness score: {best_fitness_score:.2f}")

Best fitness score: -75.33


In [18]:
pantry_consumption_df = planner.get_pantry_consumption()
pantry_consumption_df

,Ingredient,Available,Consumed,Unused,Expires in
0,chicken breast,800,0.0,800.0,6d
1,broccoli,1500,0.0,1500.0,0d
2,rice,1000,50.0,950.0,6d


In [19]:
meal_plan_df = planner.get_meal_plan_recipes()
meal_plan_df

,Day,Meal 1,Meal 2,Meal 3
0,1,Diner-Style Bacon for a Crowd,Herbed Oyster Crakers,Rustic Rub
1,2,Cranberry-Kumquat Sauce,Peanut Butter Chocolate Chip Breads,Banana-Rum Ice Cream
2,3,Buttermilk Pie Crust Dough,Fried Chickpeas,Portuguese Corn Bread
3,4,Grilled Flank Steak,"Bulgur with Leeks, Cranberries, and Almonds",Zucchini Salad With Ajo Blanco Dressing & Spic...
4,5,Peppered Lamb with Pine Nut Sauce,To Zest Citrus Fruits,Chocolate Mint Chip Parfait
5,6,Beef and Green Pepper Stir-Fry,Hazelnut and Olive Rugelach,Scandinavian Yellow Pea Soup
6,7,Hot Beef Borscht with Sour Cream,Spiced Pear Tartlets,Kid-Friendly Turkey Burgers


In [ ]:
shopping_list_df, num_ingredients, total_cost = planner.get_shopping_list()

print(f"Total ingredients needed to purchase: {num_ingredients}")
print(f"Total cost: €{total_cost:.2f}")

shopping_list_df

Total ingredients needed to purchase: 125
Total cost: €50.93


,Ingredient,Quantity to Buy (g),Cost (€)
0,(105-115ºF) water,118.3,1.18
1,Danish Canadian-style bacon (pork loin),90.7,0.91
2,Ground cloves,2.1,0.02
3,Ingredient info,100.0,1.00
4,Ketchup,50.0,0.50
...,...,...,...
121,white-wine vinegar,9.9,0.10
122,whole black peppercorns,11.1,0.11
123,whole cloves,11.1,0.11
124,zucchinis,8.0,0.08


In [21]:
daily_nutrition_df = planner.get_daily_nutrition()
daily_nutrition_df

,Calories,Protein,Δ Calories and Target Calories,Δ Protein and Target Protein
Day 1,2198.8 kcal,47.4 g,-301.2 kcal,-32.6 g
Day 2,2064.1 kcal,45.6 g,-435.9 kcal,-34.4 g
Day 3,1545.5 kcal,49.2 g,-954.5 kcal,-30.8 g
Day 4,2329.8 kcal,81.2 g,-170.2 kcal,1.2 g
Day 5,1396.6 kcal,33.8 g,-1103.4 kcal,-46.2 g
Day 6,2025.9 kcal,61.2 g,-474.1 kcal,-18.8 g
Day 7,1723.4 kcal,83.0 g,-776.6 kcal,3.0 g
